
# Geographic Demand Prediction Model

This notebook trains a **Geographic Demand Prediction model** for the pharma analytics dashboard.

### Model Details
Algorithm: Gradient Boosting Regressor  
Goal: Predict product demand by region for upcoming months  
Output: Predicted unit demand by region  
Refresh Rate: Weekly


In [ ]:

!pip install pandas scikit-learn joblib matplotlib


In [ ]:

import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor
import joblib



## Upload Dataset

Upload the dataset:
pioneer_pharma_ideal_dataset_70000_rows.csv

In [ ]:

from google.colab import files
uploaded = files.upload()


## Load Dataset

In [ ]:

df = pd.read_csv("pioneer_pharma_ideal_dataset_70000_rows.csv")

df.head()


## Convert Date Column

In [ ]:

df["order_date"] = pd.to_datetime(df["order_date"])


## Filter Only Sales Transactions

In [ ]:

df = df[df["transaction_type"] == "Sale"]


## Create Time Features

In [ ]:

df["month"] = df["order_date"].dt.month
df["year"] = df["order_date"].dt.year


## Aggregate Monthly Demand by Region and Product

In [ ]:

demand = df.groupby(
    ["region","product_name","year","month"]
)["quantity"].sum().reset_index()

demand.head()


## Encode Categorical Variables

In [ ]:

region_encoder = LabelEncoder()
product_encoder = LabelEncoder()

demand["region_encoded"] = region_encoder.fit_transform(demand["region"])
demand["product_encoded"] = product_encoder.fit_transform(demand["product_name"])


## Define Features and Target

In [ ]:

X = demand[[
    "region_encoded",
    "product_encoded",
    "year",
    "month"
]]

y = demand["quantity"]


## Train Gradient Boosting Regressor

In [ ]:

model = GradientBoostingRegressor()

model.fit(X,y)


## Save Model and Encoders

In [ ]:

joblib.dump(model,"geographic_demand_model.pkl")

joblib.dump(region_encoder,"region_encoder.pkl")

joblib.dump(product_encoder,"product_encoder.pkl")


## Download Trained Model

In [ ]:

from google.colab import files

files.download("geographic_demand_model.pkl")
files.download("region_encoder.pkl")
files.download("product_encoder.pkl")
